In [35]:
import pandas as pd

DATA_DIR=r"C:\Users\daran\Downloads\ec4aeca2a8032be0347eb12ca3d29c96\cleaned"
RAW_DIR=r"C:\Users\daran\Downloads\ec4aeca2a8032be0347eb12ca3d29c96"

In [36]:
customers = pd.read_csv(f"{DATA_DIR}\\customers_clean.csv")
billing = pd.read_csv(f"{DATA_DIR}\\billing_clean.csv")
payments = pd.read_csv(f"{DATA_DIR}\\payments_clean.csv")
subscriptions = pd.read_csv(f"{DATA_DIR}\\subscriptions_clean.csv")

contracts = pd.read_csv(f"{RAW_DIR}\\contracts.csv")
support_tickets = pd.read_csv(f"{RAW_DIR}\\support_tickets.csv")

print("Loaded.")


Loaded.


##Customer Churn Rate

In [37]:
acq_raw = customers["acquisition_date"]
parsed_dmy = pd.to_datetime(acq_raw, format="%d-%m-%Y", errors="coerce")
parsed_ymd = pd.to_datetime(acq_raw, format="%Y/%m/%d", errors="coerce")
customers["acquisition_date"] = parsed_dmy.fillna(parsed_ymd)

customers["churn_date"] = pd.to_datetime(customers["churn_date"], format="%d-%m-%Y", errors="coerce")

print("Unparseable acquisition dates:", customers["acquisition_date"].isna().sum())
print("Unparseable churn dates:", customers["churn_date"].isna().sum())

Unparseable acquisition dates: 0
Unparseable churn dates: 15494


In [38]:
period_end = customers['acquisition_date'].max().replace(day=1)
period_start = period_end - pd.DateOffset(months=1)

In [39]:
customers_at_start = customers[
    (customers['acquisition_date'] < period_start) &
    (customers['churn_date'].isna() | (customers['churn_date'] >= period_start))
].shape[0]

customers_lost = customers[
    (customers['churn_date'] >= period_start) & (customers['churn_date'] < period_end)
].shape[0]

churn_rate_kpi = (customers_lost / customers_at_start) * 100

print(f"Period: {period_start.date()} to {period_end.date()}")
print(f"Customers at start of period: {customers_at_start:,}")
print(f"Customers lost during period: {customers_lost:,}")
print(f"Customer Churn Rate: {churn_rate_kpi:.2f}%")

Period: 2026-02-01 to 2026-03-01
Customers at start of period: 15,741
Customers lost during period: 143
Customer Churn Rate: 0.91%


Customer Churn Rate — 0.91% — Within the <2.1% target, 
but this masks a worsening trend: monthly churn has climbed steadily from ~0.4% a year ago.

##Customer Retention Rate

In [40]:
churned = customers[customers["customer_status"] == "Churned"]
print("Churned customers missing churn_date:", churned["churn_date"].isna().sum())

Churned customers missing churn_date: 0


In [41]:
customers_at_start = customers[
    (customers['acquisition_date'] < period_start) &
    (customers['churn_date'].isna() | (customers['churn_date'] >= period_start))
].shape[0]

customers_at_end = customers[
    (customers['acquisition_date'] < period_end) &
    (customers['churn_date'].isna() | (customers['churn_date'] >= period_end))
].shape[0]

customers_new = customers[
    (customers['acquisition_date'] >= period_start) & (customers['acquisition_date'] < period_end)
].shape[0]

retention_rate_kpi = (customers_at_end - customers_new) / customers_at_start * 100

print(f"Customers at start: {customers_at_start:,}")
print(f"Customers at end: {customers_at_end:,}")
print(f"New customers: {customers_new:,}")
print(f"Customer Retention Rate: {retention_rate_kpi:.2f}%")

Customers at start: 15,741
Customers at end: 15,688
New customers: 90
Customer Retention Rate: 99.09%


Customer Retention Rate — 99.09%
 — Strong on paper, but it's the mirror of the same churn rate above, so it carries the same hidden upward trend in losses.

In [42]:
#ARPU

In [43]:
print(subscriptions["start_date"].dtype)
print(subscriptions["end_date"].dtype)

str
str


In [44]:
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"], format="%d-%m-%Y", errors="coerce")
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"], format="%d-%m-%Y", errors="coerce")

print("Unparseable start dates:", subscriptions["start_date"].isna().sum())
print("Unparseable end dates :", subscriptions["end_date"].isna().sum())
print(subscriptions["start_date"].dtype)

Unparseable start dates: 0
Unparseable end dates : 17017
datetime64[us]


In [45]:
print(subscriptions.columns.tolist())

['subscription_id', 'customer_id', 'plan_id', 'start_date', 'end_date', 'billing_cycle', 'monthly_charge_inr', 'subscription_status']


In [46]:
live_at_end = subscriptions[
    (subscriptions['start_date'] < period_end) &
    (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= period_end))
]

mrr = live_at_end['monthly_charge_inr'].sum()
arpu = mrr / ((customers_at_start + customers_at_end) / 2)

print(f"Live subscriptions at end of period: {len(live_at_end):,}")
print(f"MRR: Rs.{mrr:,.0f}")
print(f"ARPU: Rs.{arpu:,.2f}")

Live subscriptions at end of period: 17,151
MRR: Rs.15,087,084
ARPU: Rs.960.07


MRR / ARPU — ₹1.51 crore / ₹960.07 
— A healthy, predictable revenue base; ARPU is the benchmark the segment-level ARPUs should be compared against.

##Customer Lifetime Value

In [47]:
dupe_check = live_at_end.groupby('customer_id').size()
print("Customers with >1 active subscription:", (dupe_check > 1).sum())

Customers with >1 active subscription: 1238


In [48]:
gross_margin = 0.60  # ASSUMPTION - no gross margin figure exists in the data

clv = arpu * gross_margin * (1 / (churn_rate_kpi / 100))

print(f"Gross Margin (assumed): {gross_margin:.0%}")
print(f"Customer Lifetime Value: Rs.{clv:,.0f}")

Gross Margin (assumed): 60%
Customer Lifetime Value: Rs.63,409


Customer Lifetime Value — ₹63,409 (60% gross margin assumed)
 — Reflects the currently low churn rate; CLV is highly sensitive to churn, so it will fall quickly if the churn uptrend continues.

##Revenue Lost to Churn 

In [49]:
lost_customer_ids = customers[
    (customers['churn_date'] >= period_start) & (customers['churn_date'] < period_end)
]['customer_id']

lost_subs = subscriptions[
    subscriptions['customer_id'].isin(lost_customer_ids) &
    (subscriptions['end_date'] >= period_start) & (subscriptions['end_date'] < period_end)
]

revenue_lost = lost_subs['monthly_charge_inr'].sum()

print(f"Churned customers this period: {len(lost_customer_ids):,}")
print(f"Their subscriptions that ended: {len(lost_subs):,}")
print(f"Revenue Lost to Churn: Rs.{revenue_lost:,.0f}")

Churned customers this period: 143
Their subscriptions that ended: 143
Revenue Lost to Churn: Rs.90,782


Revenue Lost to Churn — ₹90,782 — Churned customers averaged ~₹635/month,
 noticeably below the ₹960 overall ARPU — churn is skewing toward lower-value plans this period.

Revenue Lost to Churn was calculated using each churned customer's actual subscription charge, rather than the blended company-wide ARPU, to reflect that churned customers' true revenue impact — this gave ₹90,782 for the period, versus ₹1,37,280 if a flat average ARPU were assumed.

##Monthly Recurring Revenue

In [50]:
live_at_end = subscriptions[
    (subscriptions['start_date'] < period_end) &
    (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= period_end))
]

mrr = live_at_end['monthly_charge_inr'].sum()
print(f"MRR: Rs.{mrr:,.0f}")


MRR: Rs.15,087,084


##First Contact Resolution

In [51]:
print(support_tickets["created_date"].dtype)
print(support_tickets.columns.tolist())

str
['ticket_id', 'customer_id', 'created_date', 'category', 'priority', 'channel', 'status', 'first_contact_resolution', 'assigned_employee_id', 'resolution_hours', 'resolution_date']


In [52]:
support_tickets["created_date"] = pd.to_datetime(support_tickets["created_date"], format="%d-%m-%Y", errors="coerce")
support_tickets["resolution_date"] = pd.to_datetime(support_tickets["resolution_date"], format="%d-%m-%Y", errors="coerce")

print("Unparseable created dates:", support_tickets["created_date"].isna().sum())
print("Unparseable resolution dates (nonzero OK - unresolved tickets have none):", support_tickets["resolution_date"].isna().sum())
print(support_tickets["created_date"].dtype)

Unparseable created dates: 0
Unparseable resolution dates (nonzero OK - unresolved tickets have none): 13366
datetime64[us]


In [53]:
print(support_tickets["first_contact_resolution"].value_counts(dropna=False))
print(support_tickets["status"].value_counts(dropna=False))

first_contact_resolution
No     18764
Yes    12726
Name: count, dtype: int64
status
Resolved     18124
Pending       8450
Escalated     2526
Reopened      2390
Name: count, dtype: int64


In [54]:
tickets_created = support_tickets[
    (support_tickets['created_date'] >= period_start) & (support_tickets['created_date'] < period_end)
]

fcr_rate = (tickets_created['first_contact_resolution'] == 'Yes').mean() * 100

print(f"Tickets created this period: {len(tickets_created):,}")
print(f"First Contact Resolution: {fcr_rate:.2f}%")

Tickets created this period: 187
First Contact Resolution: 42.25%


First Contact Resolution — 42.25% — Far below the >78% target; 
this is the weakest metric in the whole set and a direct, actionable churn lever to flag.

##Avg Resolution Time

In [55]:
tickets_resolved = support_tickets[
    (support_tickets['status'] == 'Resolved') &
    (support_tickets['resolution_date'] >= period_start) & (support_tickets['resolution_date'] < period_end)
]

avg_resolution_hours = tickets_resolved['resolution_hours'].mean()

print(f"Tickets resolved this period: {len(tickets_resolved):,}")
print(f"Avg Resolution Time: {avg_resolution_hours:.1f} hours")

Tickets resolved this period: 112
Avg Resolution Time: 15.4 hours


resolution_hours was used directly rather than recalculating from created_date/resolution_date, since those date fields only carry day-level precision and would understate same-day resolutions.

Avg Resolution Time — 15.4 hours
 — Comfortably under the <24h target; support speed isn't the problem, first-contact effectiveness is.

##Contract Renewal Rate

In [56]:
print(contracts["end_date"].dtype)
print(contracts.columns.tolist())
print(contracts["renewal_status"].value_counts(dropna=False))

str
['contract_id', 'customer_id', 'plan_id', 'contract_type', 'contract_length_months', 'start_date', 'end_date', 'auto_renew', 'renewal_status']
renewal_status
Renewed         3676
Auto-Renewed    2266
Active          1829
Pending         1360
Not Renewed     1058
Cancelled        418
Name: count, dtype: int64


In [57]:
contracts["start_date"] = pd.to_datetime(contracts["start_date"], format="%d-%m-%Y", errors="coerce")
contracts["end_date"] = pd.to_datetime(contracts["end_date"], format="%d-%m-%Y", errors="coerce")

print("Unparseable end dates:", contracts["end_date"].isna().sum())

Unparseable end dates: 0


In [58]:
renewed = ["Renewed", "Auto-Renewed"]
decided = renewed + ["Not Renewed", "Cancelled"]  # Active/Pending are undecided - excluded

contracts_due = contracts[
    (contracts['end_date'] >= period_start) & (contracts['end_date'] < period_end) &
    (contracts['renewal_status'].isin(decided))
]

renewal_rate = contracts_due['renewal_status'].isin(renewed).mean() * 100

print(f"Contracts due for renewal this period: {len(contracts_due):,}")
print(f"Contract Renewal Rate: {renewal_rate:.2f}%")

Contracts due for renewal this period: 38
Contract Renewal Rate: 65.79%


Contract Renewal Rate — 65.79% — Well below the >88% target for postpaid/fibre, 
though based on a small sample (38 contracts) this period — worth checking against a longer window before treating it as definitive.

In [59]:
print(customers.columns.tolist())

['customer_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'age', 'email', 'phone_number', 'address', 'city_id', 'city_name', 'state_id', 'pin_code', 'city_tier', 'region_id', 'occupation', 'annual_income_inr', 'customer_segment', 'product_line', 'plan_id', 'acquisition_channel', 'acquisition_date', 'tenure_months', 'customer_status', 'churn_date', 'dob_parsed', 'calculated_age', 'dob_test']


In [60]:
active_at_end = customers[
    (customers['acquisition_date'] < period_end) &
    (customers['churn_date'].isna() | (customers['churn_date'] >= period_end))
]

avg_tenure = active_at_end['tenure_months'].mean()

print(f"Active customers at end of period: {len(active_at_end):,}")
print(f"Avg Customer Tenure: {avg_tenure:.1f} months")

Active customers at end of period: 15,688
Avg Customer Tenure: 86.2 months


Avg Customer Tenure — 86.2 months (~7.2 years) —
 Very high tenure given the low churn rate, consistent with a mature, long-standing customer base.

In [61]:
kpi_summary = pd.DataFrame({
    "KPI": ["Churn Rate %", "Retention Rate %", "MRR (INR)", "ARPU (INR)", "CLV (INR)",
            "Revenue Lost to Churn (INR)", "First Contact Resolution %", "Avg Resolution Time (hrs)",
            "Contract Renewal Rate %", "Avg Customer Tenure (months)"],
    "Value": [churn_rate_kpi, retention_rate_kpi, mrr, arpu, clv,
              revenue_lost, fcr_rate, avg_resolution_hours, renewal_rate, avg_tenure],
    "Target": ["< 2.1%", "n/a", "n/a", "n/a", "n/a", "n/a", "> 78%", "< 24h", "> 88%", "n/a"]
})
kpi_summary

,KPI,Value,Target
0,Churn Rate %,0.91,< 2.1%
1,Retention Rate %,99.09,n/a
2,MRR (INR),"15,087,084.00",n/a
3,ARPU (INR),960.07,n/a
4,CLV (INR),"63,409.20",n/a
5,Revenue Lost to Churn (INR),"90,782.00",n/a
6,First Contact Resolution %,42.25,> 78%
7,Avg Resolution Time (hrs),15.38,< 24h
8,Contract Renewal Rate %,65.79,> 88%
9,Avg Customer Tenure (months),86.22,n/a


In [62]:
pd.set_option("display.float_format", "{:,.2f}".format)
kpi_summary_display = kpi_summary.copy()
kpi_summary_display["Value"] = kpi_summary_display["Value"].round(2)
kpi_summary_display

,KPI,Value,Target
0,Churn Rate %,0.91,< 2.1%
1,Retention Rate %,99.09,n/a
2,MRR (INR),"15,087,084.00",n/a
3,ARPU (INR),960.07,n/a
4,CLV (INR),"63,409.20",n/a
5,Revenue Lost to Churn (INR),"90,782.00",n/a
6,First Contact Resolution %,42.25,> 78%
7,Avg Resolution Time (hrs),15.38,< 24h
8,Contract Renewal Rate %,65.79,> 88%
9,Avg Customer Tenure (months),86.22,n/a


##Cohort Retention analysis

In [63]:
customers['cohort'] = customers['acquisition_date'].dt.to_period('M')

def retained_after(df, months):
    checkpoint = df['acquisition_date'] + pd.DateOffset(months=months)
    still_active = df['churn_date'].isna() | (df['churn_date'] >= checkpoint)
    observable = checkpoint <= period_end  # can't know the answer for cohorts too young yet
    return still_active.where(observable)  # NaN if not old enough to check

customers['active_3m'] = retained_after(customers, 3)
customers['active_6m'] = retained_after(customers, 6)
customers['active_12m'] = retained_after(customers, 12)

cohort_table = customers.pivot_table(
    index='cohort',
    values=['active_3m', 'active_6m', 'active_12m'],
    aggfunc='mean'
) * 100

cohort_table['cohort_size'] = customers.groupby('cohort').size()
cohort_table.round(1).tail(15)

,active_12m,active_3m,active_6m,cohort_size
cohort,,,,
2024-10,76.07,94.02,88.03,117
2024-11,74.56,96.49,92.98,114
2024-12,78.18,92.73,87.27,110
2025-01,68.22,96.26,85.98,107
2025-02,66.37,90.27,81.42,113
2025-03,100.00,91.74,84.30,121
2025-04,NaN,92.93,84.85,99
2025-05,NaN,94.69,86.73,113
2025-06,NaN,79.63,74.07,108


In [64]:
mar25 = customers[customers['cohort'] == '2025-03']
print("Cohort size:", len(mar25))
print("Churned in this cohort:", (mar25['customer_status'] == 'Churned').sum())
print(mar25[['acquisition_date', 'churn_date', 'customer_status']].head(10))

Cohort size: 121
Churned in this cohort: 35
     acquisition_date churn_date customer_status
154        2025-03-29        NaT          Active
222        2025-03-14        NaT          Active
360        2025-03-29 2026-03-01         Churned
463        2025-03-19        NaT          Active
483        2025-03-22 2025-07-29         Churned
646        2025-03-04 2025-04-06         Churned
721        2025-03-21        NaT          Active
776        2025-03-19 2025-03-29         Churned
1002       2025-03-16 2025-07-29         Churned
1409       2025-03-23 2025-06-30         Churned


In [65]:
def retained_after(df, months):
    checkpoint = df['acquisition_date'] + pd.DateOffset(months=months)
    still_active = df['churn_date'].isna() | (df['churn_date'] >= checkpoint)
    observable = checkpoint <= period_end
    return still_active.where(observable)

for m in [3, 6, 12]:
    customers[f'active_{m}m'] = retained_after(customers, m)

agg = {f'active_{m}m': ['mean', 'count'] for m in [3, 6, 12]}
cohort_table = customers.pivot_table(index='cohort', aggfunc=agg)
cohort_table.columns = ['_'.join(col) for col in cohort_table.columns]
for m in [3, 6, 12]:
    cohort_table[f'active_{m}m_mean'] = (cohort_table[f'active_{m}m_mean'] * 100).round(1)
cohort_table['cohort_size'] = customers.groupby('cohort').size()
cohort_table.tail(15)

,active_12m_count,active_12m_mean,active_3m_count,active_3m_mean,active_6m_count,active_6m_mean,cohort_size
cohort,,,,,,,
2025-01,107,68.20,107,96.30,107,86.00,107
2025-02,113,66.40,113,90.30,113,81.40,113
2025-03,2,100.00,121,91.70,121,84.30,121
2025-04,0,NaN,99,92.90,99,84.80,99
2025-05,0,NaN,113,94.70,113,86.70,113
2025-06,0,NaN,108,79.60,108,74.10,108
2025-07,0,NaN,125,88.00,125,76.80,125
2025-08,0,NaN,125,89.60,125,73.60,125
2025-09,0,NaN,113,81.40,6,83.30,113


In [66]:
for m in [3, 6, 12]:
    coverage = cohort_table[f'active_{m}m_count'] / cohort_table['cohort_size']
    cohort_table[f'active_{m}m_mean'] = cohort_table[f'active_{m}m_mean'].where(coverage >= 0.8)

cohort_table[['cohort_size', 'active_3m_mean', 'active_6m_mean', 'active_12m_mean']].tail(15)

,cohort_size,active_3m_mean,active_6m_mean,active_12m_mean
cohort,,,,
2025-01,107,96.30,86.00,68.20
2025-02,113,90.30,81.40,66.40
2025-03,121,91.70,84.30,NaN
2025-04,99,92.90,84.80,NaN
2025-05,113,94.70,86.70,NaN
2025-06,108,79.60,74.10,NaN
2025-07,125,88.00,76.80,NaN
2025-08,125,89.60,73.60,NaN
2025-09,113,81.40,NaN,NaN


Cohort Retention — 3-month retention has declined from ~95% (early-2025 cohorts) to ~80–90% (mid/late-2025 cohorts), and 6-month retention shows the same pattern
 — independent confirmation that retention has weakened since mid-2025, not just a blip in the monthly churn number.

In [67]:
def month_kpis(month):
    start = pd.Period(month, "M").start_time
    end = (pd.Period(month, "M") + 1).start_time

    at_start = customers[(customers['acquisition_date'] < start) &
                         (customers['churn_date'].isna() | (customers['churn_date'] >= start))]
    at_end = customers[(customers['acquisition_date'] < end) &
                       (customers['churn_date'].isna() | (customers['churn_date'] >= end))]
    new = customers[(customers['acquisition_date'] >= start) & (customers['acquisition_date'] < end)]
    lost = customers[(customers['churn_date'] >= start) & (customers['churn_date'] < end)]

    live = subscriptions[(subscriptions['start_date'] < end) &
                         (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= end))]
    lost_subs = subscriptions[subscriptions['customer_id'].isin(lost['customer_id']) &
                              (subscriptions['end_date'] >= start) & (subscriptions['end_date'] < end)]

    return {
        "month": start,
        "customers_start": len(at_start),
        "customers_end": len(at_end),
        "new_customers": len(new),
        "customers_lost": len(lost),
        "churn_rate_pct": len(lost) / len(at_start) * 100,
        "retention_rate_pct": (len(at_end) - len(new)) / len(at_start) * 100,
        "mrr_inr": live['monthly_charge_inr'].sum(),
        "revenue_lost_inr": lost_subs['monthly_charge_inr'].sum(),
    }

months = pd.period_range(end=period_start.to_period("M"), periods=12, freq="M")
monthly_kpis = pd.DataFrame([month_kpis(str(m)) for m in months])
print(monthly_kpis.round(2))

        month  customers_start  customers_end  new_customers  customers_lost  \
0  2025-03-01            15466          15515            121              72   
1  2025-04-01            15515          15546             99              68   
2  2025-05-01            15546          15596            113              63   
3  2025-06-01            15596          15627            108              77   
4  2025-07-01            15627          15661            125              91   
5  2025-08-01            15661          15684            125             102   
6  2025-09-01            15684          15714            113              83   
7  2025-10-01            15714          15755            135              94   
8  2025-11-01            15755          15755            113             113   
9  2025-12-01            15755          15752            106             109   
10 2026-01-01            15752          15741            113             124   
11 2026-02-01            15741          

C:\Users\daran\AppData\Local\Temp\ipykernel_20832\3491913907.py:31: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(monthly_kpis.round(2))


In [68]:
plans = pd.read_csv(f"{RAW_DIR}\\plans.csv")

cust_export = customers.merge(plans[['plan_id', 'monthly_charge']], on='plan_id', how='left')
cust_export['churned_flag'] = (cust_export['customer_status'] == 'Churned').astype(int)
cust_export['tenure_band'] = pd.cut(cust_export['tenure_months'],
                                    bins=[-1, 12, 36, 72, 1000],
                                    labels=['0-12', '13-36', '37-72', '73+'])

# drop personal details and leftover debugging columns the dashboard doesn't need
drop_cols = ['first_name', 'last_name', 'email', 'phone_number', 'address', 'date_of_birth',
             'dob_parsed', 'calculated_age', 'dob_test', 'cohort',
             'active_3m', 'active_6m', 'active_12m']
cust_export = cust_export.drop(columns=[c for c in drop_cols if c in cust_export.columns])

import os
OUT = os.path.join(RAW_DIR, "powerbi_data")
os.makedirs(OUT, exist_ok=True)
monthly_kpis.to_csv(os.path.join(OUT, "monthly_kpis.csv"), index=False)
cust_export.to_csv(os.path.join(OUT, "customers_powerbi.csv"), index=False)
support_tickets.to_csv(os.path.join(OUT, "support_tickets_powerbi.csv"), index=False)
print("Saved to", OUT, "|", cust_export.shape)

Saved to C:\Users\daran\Downloads\ec4aeca2a8032be0347eb12ca3d29c96\powerbi_data | (19000, 22)
